# Reinforcement Learning

# 4. Online control

This notebook presents the **online control** of an agent by SARSA and Q-learning.

Please read the instructions:
* Write concise code and text.
* Check that your notebook runs without errors and in reasonable time (< 3 min).
* Clear all cell outputs before upload on Moodle.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from model import TicTacToe, Nim, ConnectFour
from agent import Agent, OnlineControl
from dynamic import PolicyEvaluation, ValueIteration

## To do

* Complete the method ``train``of the class ``SARSA`` and test it on Tic-Tac-Toe.
* Plot the expected return with respect to the number of episodes used for training (e.g., from 1 to 1000).
* What is the first move of the agent after training (without randomization)?


## SARSA

In [ ]:
class SARSA(OnlineControl):
    """Online control by SARSA."""
        
    def train(self, state=None, horizon=100, epsilon=0.1):
        """Learn the action-value function online.
        
        Parameters
        ----------
        state :
            First state of the episode (default = initial state of the environment).
        horizon : int
            Time horizon of the episode.
        epsilon : float
            Exploration rate.
        """
        self.environment.reset(state)
        state = self.environment.state
        if not self.environment.is_terminal(state):
            action = self.randomize_best_action(state, epsilon=epsilon)
            for t in range(horizon):
                code = self.environment.encode(state)
                self.action_count[code][action] += 1
                reward, stop = self.environment.step(action)
                # to be modified (get sample)
                # begin
                next_state = self.environment.state
                if stop:
                    sample = reward
                else:
                    next_action = self.randomize_best_action(next_state, epsilon=epsilon)
                    next_code = self.environment.encode(next_state)
                    sample = reward + self.gamma * self.action_value[next_code][next_action]
                # end
                diff = sample - self.action_value[code][action]
                count = self.action_count[code][action]
                self.action_value[code][action] += diff / count
                if stop:
                    break
                # to be modified (update state and action)
                # begin
                state = next_state
                action = next_action
                # end

In [ ]:
Game = TicTacToe
game = Game()
agent = SARSA(game)

In [ ]:
returns = agent.get_returns()
np.unique(returns, return_counts=True)

In [ ]:
n_episodes = 500
for i in range(n_episodes):
    agent.train()

In [ ]:
agent.policy = agent.improve_policy()
returns = agent.get_returns()
np.unique(returns, return_counts=True)

In [ ]:
Game = TicTacToe
game = Game()
agent = SARSA(game)

N_TRAIN = 1000    
EVAL_EVERY = 10     
N_EVAL = 200        
episodes = []
expected_returns = []

for n in range(1, N_TRAIN + 1):
    agent.train(horizon=100, epsilon=0.1)

    if n % EVAL_EVERY == 0:
        agent.policy = agent.improve_policy()  
        returns = agent.get_returns(n_episodes=N_EVAL)
        episodes.append(n)
        expected_returns.append(np.mean(returns))

plt.figure(figsize=(7, 4))
plt.plot(episodes, expected_returns)
plt.title(r"Expected return E[G] vs training episodes (SARSA, TicTacToe)")
plt.xlabel("Training episodes")
plt.ylabel(r"Expected return E[G]")
plt.grid(True)
plt.show()

In [ ]:
agent.environment.reset()
start_state = agent.environment.state
best_actions = agent.get_best_actions(start_state)
first_move = min(best_actions)
print("Best actions from start =", best_actions)
print("First move (deterministic) =", first_move)

## To do

* Do the same for the class ``QLearning``.
* Compare SARSA and Q-learning on Tic-Tac-Toe (play first) and Nim (play second) with a random adversary, then a perfect adversary.<br> Comment the results.
* Test SARSA and Q-learning on Connect 4 against a random adversary. <br>Comment the results.

## Q-learning

In [ ]:
class QLearning(OnlineControl):
    """Online control by Q-learning."""
        
    def train(self, state=None, horizon=100, epsilon=0.1):
        """Learn the action-value function online.
        
        Parameters
        ----------
        state :
            First state of the episode (default = initial state of the environment).
        horizon : int
            Time horizon of the episode.
        epsilon : float
            Exploration rate.
        """
        self.environment.reset(state)
        state = self.environment.state
        # to be completed
        if self.environment.is_terminal(state):
            return

        for t in range(horizon):
            action = self.randomize_best_action(state, epsilon=epsilon)

            code = self.environment.encode(state)
            self.action_count[code][action] += 1

            reward, stop = self.environment.step(action)
            next_state = self.environment.state

            if stop:
                sample = reward
            else:
                next_code = self.environment.encode(next_state)
                next_actions = self.get_actions(next_state)
                best_next_value = max(self.action_value[next_code][a] for a in next_actions)
                sample = reward + self.gamma * best_next_value

            diff = sample - self.action_value[code][action]
            count = self.action_count[code][action]
            self.action_value[code][action] += diff / count

            if stop:
                break

            state = next_state

In [ ]:
def train_agent(agent, n_train=5000, horizon=100, epsilon=0.1):
    for _ in range(n_train):
        agent.train(horizon=horizon, epsilon=epsilon)
    agent.policy = agent.improve_policy()

def evaluate_agent(agent, n_eval=500):
    returns = agent.get_returns(n_episodes=n_eval)
    mean_return = float(np.mean(returns))
    dist = dict(zip(*np.unique(returns, return_counts=True)))
    return mean_return, dist

In [ ]:
N_TRAIN = 5000
N_EVAL  = 200
EPS = 0.1
HORIZON = 9

game_ttt = TicTacToe(adversary_policy='random', play_first=True)

sarsa_ttt = SARSA(game_ttt)
train_agent(sarsa_ttt, n_train=N_TRAIN, horizon=HORIZON, epsilon=EPS)
mean_sarsa, dist_sarsa = evaluate_agent(sarsa_ttt, n_eval=N_EVAL)

game_ttt2 = TicTacToe(adversary_policy='random', play_first=True)  
q_ttt = QLearning(game_ttt2)
train_agent(q_ttt, n_train=N_TRAIN, horizon=HORIZON, epsilon=EPS)
mean_q, dist_q = evaluate_agent(q_ttt, n_eval=N_EVAL)

print("=== TicTacToe vs RANDOM adversary (play first) ===")
print("SARSA     mean return =", mean_sarsa, "dist =", dist_sarsa)
print("Q-learning mean return =", mean_q,     "dist =", dist_q)

In [ ]:
N_TRAIN = 5000
N_EVAL  = 200
EPS = 0.1
HORIZON = 50

game_nim = Nim(adversary_policy='random', play_first=False) 
sarsa_nim = SARSA(game_nim)
train_agent(sarsa_nim, n_train=N_TRAIN, horizon=HORIZON, epsilon=EPS)
mean_sarsa, dist_sarsa = evaluate_agent(sarsa_nim, n_eval=N_EVAL)

game_nim2 = Nim(adversary_policy='random', play_first=False)
q_nim = QLearning(game_nim2)
train_agent(q_nim, n_train=N_TRAIN, horizon=HORIZON, epsilon=EPS)
mean_q, dist_q = evaluate_agent(q_nim, n_eval=N_EVAL)

print("=== Nim vs RANDOM adversary (play second) ===")
print("SARSA     mean return =", mean_sarsa, "dist =", dist_sarsa)
print("Q-learning mean return =", mean_q,     "dist =", dist_q)

In [ ]:
game_ttt = TicTacToe(adversary_policy='one_step', play_first=True)
sarsa = SARSA(game_ttt); train_agent(sarsa, n_train=N_TRAIN, horizon=9, epsilon=0.1)
m1, d1 = evaluate_agent(sarsa, n_eval=N_EVAL)

game_ttt2 = TicTacToe(adversary_policy='one_step', play_first=True)
q = QLearning(game_ttt2); train_agent(q, n_train=N_TRAIN, horizon=9,epsilon=0.1)
m2, d2 = evaluate_agent(q, n_eval=N_EVAL)

print("=== TicTacToe vs ONE_STEP adversary ===")
print("SARSA     mean return =", m1, "dist =", d1)
print("Q-learning mean return =", m2, "dist =", d2)

game_nim = Nim(adversary_policy='one_step', play_first=False)
sarsa = SARSA(game_nim); train_agent(sarsa, n_train=N_TRAIN, horizon=HORIZON, epsilon=0.1)
m1, d1 = evaluate_agent(sarsa, n_eval=N_EVAL)

game_nim2 = Nim(adversary_policy='one_step', play_first=False)
q = QLearning(game_nim2); train_agent(q, n_train=N_TRAIN, horizon=HORIZON, epsilon=0.1)
m2, d2 = evaluate_agent(q, n_eval=N_EVAL)

print("=== Nim vs ONE_STEP adversary (play second)===")
print("SARSA     mean return =", m1, "dist =", d1)
print("Q-learning mean return =", m2, "dist =", d2)

ANSWER:

Against a random adversary, both SARSA and Q-learning perform well. Q-learning is usually slightly better with more wins and higher mean return. Against one_step, performance drops with more draws and fewer easy wins. But, both remain strong: differences are small, with Q-learning often still a bit ahead.

In [ ]:
N_TRAIN = 800
N_EVAL  = 50
EPS     = 0.1
HORIZON = 42

game_c4 = ConnectFour(adversary_policy='random', play_first=True)
sarsa_c4 = SARSA(game_c4)
train_agent(sarsa_c4, n_train=N_TRAIN, horizon=HORIZON, epsilon=EPS)
mean_sarsa, dist_sarsa = evaluate_agent(sarsa_c4, n_eval=N_EVAL)

game_c4b = ConnectFour(adversary_policy='random', play_first=True)
q_c4 = QLearning(game_c4b)
train_agent(q_c4, n_train=N_TRAIN, horizon=HORIZON, epsilon=EPS)
mean_q, dist_q = evaluate_agent(q_c4, n_eval=N_EVAL)

print("=== ConnectFour vs RANDOM adversary ===")
print("SARSA     mean return =", mean_sarsa, "dist =", dist_sarsa)
print("Q-learning mean return =", mean_q,     "dist =", dist_q)

ANSWER:

Both SARSA and Q-learning are much weaker than in Tic-Tac-Toe because the problem is far larger. With limited training they don’t learn enough. Q-learning tends to outperform SARSA, but neither is reliable without many more episodes (and/or better exploration).